In [1]:
import pandas as pd


In [2]:

# Load data
users = pd.read_csv("Final_Updated_Expanded_Users.csv")
destinations = pd.read_csv("Expanded_Destinations.csv")
reviews = pd.read_csv("Final_Updated_Expanded_Reviews.csv")
history = pd.read_csv("Final_Updated_Expanded_UserHistory.csv")

# 1 data handling

### 1.1, check data

In [9]:
def check_data(df, name):
    print(f"\n📊 Dataset: {name}")
    print("Shape:", df.shape)
    print("\nMissing values:\n", df.isnull().sum())
    print("\nDuplicates:", df.duplicated().sum())
    print("\nData types:\n", df.dtypes)
    print("\nSample:\n", df.head())


In [7]:
check_data(users, "Users")


📊 Dataset: Users
Shape: (999, 7)

Missing values:
 UserID              0
Name                0
Email               0
Preferences         0
Gender              0
NumberOfAdults      0
NumberOfChildren    0
dtype: int64

Duplicates: 0

Data types:
 UserID              int64
Name                  str
Email                 str
Preferences           str
Gender                str
NumberOfAdults      int64
NumberOfChildren    int64
dtype: object

Sample:
    UserID   Name              Email          Preferences  Gender  \
0       1  Kavya  kavya@example.com  Beaches, Historical  Female   
1       2  Rohan  rohan@example.com    Nature, Adventure    Male   
2       3  Kavya  kavya@example.com     City, Historical  Female   
3       4  Anika  anika@example.com  Beaches, Historical  Female   
4       5  Tanvi  tanvi@example.com    Nature, Adventure  Female   

   NumberOfAdults  NumberOfChildren  
0               1                 0  
1               2                 2  
2               2      

In [4]:
check_data(destinations, "Destinations")


📊 Dataset: Destinations
Shape: (1000, 6)

Missing values:
 DestinationID      0
Name               0
State              0
Type               0
Popularity         0
BestTimeToVisit    0
dtype: int64

Duplicates: 0

Data types:
 DestinationID        int64
Name                   str
State                  str
Type                   str
Popularity         float64
BestTimeToVisit        str
dtype: object

Sample:
    DestinationID               Name              State        Type  \
0              1          Taj Mahal      Uttar Pradesh  Historical   
1              2        Goa Beaches                Goa       Beach   
2              3        Jaipur City          Rajasthan        City   
3              4  Kerala Backwaters             Kerala      Nature   
4              5         Leh Ladakh  Jammu and Kashmir   Adventure   

   Popularity BestTimeToVisit  
0    8.691906         Nov-Feb  
1    8.605032         Nov-Mar  
2    9.225372         Oct-Mar  
3    7.977386         Sep-Mar  
4    

In [5]:
check_data(reviews, "Reviews")


📊 Dataset: Reviews
Shape: (999, 5)

Missing values:
 ReviewID         0
DestinationID    0
UserID           0
Rating           0
ReviewText       0
dtype: int64

Duplicates: 0

Data types:
 ReviewID         int64
DestinationID    int64
UserID           int64
Rating           int64
ReviewText         str
dtype: object

Sample:
    ReviewID  DestinationID  UserID  Rating            ReviewText
0         1            178     327       2  Incredible monument!
1         2            411     783       1    Loved the beaches!
2         3            927      12       2   A historical wonder
3         4            358     959       3  Incredible monument!
4         5            989     353       2    Loved the beaches!


In [6]:
check_data(history, "UserHistory")


📊 Dataset: UserHistory
Shape: (999, 5)

Missing values:
 HistoryID           0
UserID              0
DestinationID       0
VisitDate           0
ExperienceRating    0
dtype: int64

Duplicates: 0

Data types:
 HistoryID           int64
UserID              int64
DestinationID       int64
VisitDate             str
ExperienceRating    int64
dtype: object

Sample:
    HistoryID  UserID  DestinationID   VisitDate  ExperienceRating
0          1     525            760  2024-01-01                 3
1          2     184            532  2024-02-15                 5
2          3     897            786  2024-03-20                 2
3          4     470            660  2024-01-01                 1
4          5     989            389  2024-02-15                 4


⇨ Based on the results above, the dataset appears to be well-preprocessed, with no missing values or duplicates detected. Therefore, only a few additional and straightforward steps are required to finalize the data cleaning process before moving on to feature engineering and modeling.

### 1.2, Clean data

In [10]:
# Convert data types
if "VisitDate" in history.columns:
    history["VisitDate"] = pd.to_datetime(history["VisitDate"], errors='coerce')

In [13]:
# --- Remove invalid ratings

# Reviews
invalid_reviews = reviews[~reviews["Rating"].between(1, 5)]

if len(invalid_reviews) > 0:
    print(f" Found {len(invalid_reviews)} invalid ratings in Reviews → removing...")
    reviews = reviews[reviews["Rating"].between(1, 5)]
else:
    print(" No invalid ratings found in Reviews")

# UserHistory
if "ExperienceRating" in history.columns:
    invalid_history = history[~history["ExperienceRating"].between(1, 5)]
    
    if len(invalid_history) > 0:
        print(f" Found {len(invalid_history)} invalid ratings in UserHistory → removing...")
        history = history[history["ExperienceRating"].between(1, 5)]
    else:
        print(" No invalid ratings found in UserHistory")

 No invalid ratings found in Reviews
 No invalid ratings found in UserHistory


In [14]:
# Remove rows with invalid UserID or DestinationID

valid_users = set(users["UserID"])
valid_dest = set(destinations["DestinationID"])

# --- Reviews
invalid_reviews_fk = reviews[
    ~reviews["UserID"].isin(valid_users) |
    ~reviews["DestinationID"].isin(valid_dest)
]

if len(invalid_reviews_fk) > 0:
    print(f" Found {len(invalid_reviews_fk)} invalid foreign keys in Reviews → removing...")
    reviews = reviews[
        reviews["UserID"].isin(valid_users) &
        reviews["DestinationID"].isin(valid_dest)
    ]
else:
    print(" No invalid foreign keys found in Reviews")

# --- UserHistory
invalid_history_fk = history[
    ~history["UserID"].isin(valid_users) |
    ~history["DestinationID"].isin(valid_dest)
]

if len(invalid_history_fk) > 0:
    print(f" Found {len(invalid_history_fk)} invalid foreign keys in UserHistory → removing...")
    history = history[
        history["UserID"].isin(valid_users) &
        history["DestinationID"].isin(valid_dest)
    ]
else:
    print(" No invalid foreign keys found in UserHistory")

 No invalid foreign keys found in Reviews
 No invalid foreign keys found in UserHistory


⇨ No invalid ratings and foreign keys were found, indicating good data consistency across all datasets.

# 2, EDA

#  Exploratory Data Analysis (EDA)
# Common and Dataset-Specific Features

---

## 1. Common Features Across Datasets

| Feature | Dataset(s) | Role |
|--------|-----------|------|
| UserID | Users, Reviews, UserHistory | User identifier |
| DestinationID | Destinations, Reviews, UserHistory | Item identifier |
| Rating | Reviews, UserHistory (ExperienceRating) | Interaction signal |

### Explanation
These common features act as **linking keys and interaction signals** across all datasets. They enable the construction of relationships between users and destinations.

 **Insights:**
- `UserID` + `DestinationID` → forms the user-item interaction matrix  
- `Rating` → used for collaborative filtering models  

---

## 2. Users Dataset – Specific Features

| Feature | Type | Description |
|--------|------|------------|
| Name | Categorical | User name |
| Email | Categorical | Email address |
| Preferences | Categorical | Travel preferences |
| Gender | Categorical | Gender |
| NumberOfAdults | Numeric | Number of adults |
| NumberOfChildren | Numeric | Number of children |

### Explanation
These features describe user demographics and travel context.

 **Insights:**
- Enables segmentation: solo travelers, couples, families  
- Supports personalized recommendations  

---

## 3. Destinations Dataset – Specific Features

| Feature | Type | Description |
|--------|------|------------|
| Name | Categorical | Destination name |
| State | Categorical | Location |
| Type | Categorical | Destination type |
| Popularity | Numeric | Popularity score |
| BestTimeToVisit | Categorical | Best visiting time |

### Explanation
These features represent item characteristics.

 **Insights:**
- `Type` → key feature for content-based filtering  
- `Popularity` → useful for baseline recommendation  

---

## 4. Reviews Dataset – Specific Features

| Feature | Type | Description |
|--------|------|------------|
| ReviewID | ID | Review identifier |
| Rating | Numeric | User rating (1–5) |
| ReviewText | Text | Review content |

### Explanation
Reviews provide **explicit feedback** from users.

 **Insights:**
- Rating → core signal for collaborative filtering  
- Text → can be used for sentiment analysis (optional)  

---

## 5. UserHistory Dataset – Specific Features

| Feature | Type | Description |
|--------|------|------------|
| HistoryID | ID | History identifier |
| VisitDate | Datetime | Visit date |
| ExperienceRating | Numeric | Experience rating |

### Explanation
UserHistory captures **actual user behavior**.

 **Insights:**
- More reliable than reviews (reflects real actions)  
- Enables time-based or sequential recommendation  

---

## 6. Dataset Roles Summary

| Dataset | Role |
|--------|------|
| Users | User profile |
| Destinations | Items |
| Reviews | Explicit feedback |
| UserHistory | Implicit feedback |

---

## 7. Key Insights

- The dataset contains both **explicit and implicit feedback**  
- Clear separation between user, item, and interaction data  
- Well-suited for **hybrid recommendation systems**  

---

## 8. Conclusion

The dataset structure is well-designed, with strong relationships between entities. It provides sufficient information to build scalable and personalized recommendation models.